# Data Loading into MongoDB

First we start a new Spark session where we load the whole dataset from the CSV file.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
    .appName("Data Loading")\
    .getOrCreate()



Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/30 09:18:41 WARN Utils: Your hostname, MacBook-Air-de-Martin.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.2 instead (on interface en0)
26/04/30 09:18:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/30 09:18:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
csv_path = "DATA/synthetic_student_learning_dataset_10000.csv"
df = spark.read.option("header", True).option("inferSchema", True).csv(csv_path)

In [3]:
df.printSchema()

root
 |-- respondent_id: integer (nullable = true)
 |-- education_level: string (nullable = true)
 |-- study_hours_per_day: integer (nullable = true)
 |-- preferred_learning_method: string (nullable = true)
 |-- main_learning_challenge: string (nullable = true)
 |-- motivation_level: string (nullable = true)
 |-- online_learning_opinion: string (nullable = true)
 |-- device_used_for_study: string (nullable = true)



In [4]:
for row in df.limit(5).collect():
    print(row)

Row(respondent_id=1, education_level='Undergraduate', study_hours_per_day=6, preferred_learning_method='Practice exercises', main_learning_challenge='Time management difficulty', motivation_level='Low', online_learning_opinion='Online learning is flexible and convenient', device_used_for_study='Tablet')
Row(respondent_id=2, education_level='Postgraduate', study_hours_per_day=1, preferred_learning_method='Practice exercises', main_learning_challenge='Academic workload pressure', motivation_level='High', online_learning_opinion='Online learning is effective for theory-based subjects', device_used_for_study='Laptop')
Row(respondent_id=3, education_level='Graduate', study_hours_per_day=1, preferred_learning_method='Interactive discussion', main_learning_challenge='Lack of concentration', motivation_level='Low', online_learning_opinion='Online learning is effective for theory-based subjects', device_used_for_study='Laptop')
Row(respondent_id=4, education_level='Graduate', study_hours_per_da

Now trying to process the free text fields so that we can get the most common words used as an opinion (stopwords removed) to see if there are any interesting patterns in the language used

In [19]:
from pyspark.sql.functions import split, explode, col, lower, regexp_replace
opinion_words = df.select(
                    explode(
                        split(col("online_learning_opinion"), r"\s+") #we use <r"\s+"> instead of <" "> to split on any whitespace character(s), not just spaces.
                    ).alias("opinion")
                )
opinion_words.show()

+------------+
|     opinion|
+------------+
|      Online|
|    learning|
|          is|
|    flexible|
|         and|
|  convenient|
|      Online|
|    learning|
|          is|
|   effective|
|         for|
|theory-based|
|    subjects|
|      Online|
|    learning|
|          is|
|   effective|
|         for|
|theory-based|
|    subjects|
+------------+
only showing top 20 rows


We can group the words here, however, there is more cleaning to be done before we have a relevant set of used words:

In [37]:
first_word_count = opinion_words.groupBy("opinion").count().orderBy("count", ascending=False)
first_word_count.show(20)

+------------+-----+
|     opinion|count|
+------------+-----+
|      Online| 8041|
|    learning| 7909|
|          is| 5951|
|   effective| 3975|
|      access| 2091|
|     diverse| 2091|
|     provide| 2091|
|   resources| 2091|
|   platforms| 2091|
|          to| 2091|
|         for| 2016|
|theory-based| 2016|
|    subjects| 2016|
|    flexible| 1976|
|         and| 1976|
|  convenient| 1976|
|      online| 1959|
|        more| 1959|
|        than| 1959|
|     Blended| 1959|
+------------+-----+
only showing top 20 rows



Now we make everything lower case, so that the grouping is more accurate.

In [26]:
opinion_words_lower = opinion_words.select(lower(col("opinion")).alias("opinion"))
opinion_words_lower.show()

+------------+
|     opinion|
+------------+
|      online|
|    learning|
|          is|
|    flexible|
|         and|
|  convenient|
|      online|
|    learning|
|          is|
|   effective|
|         for|
|theory-based|
|    subjects|
|      online|
|    learning|
|          is|
|   effective|
|         for|
|theory-based|
|    subjects|
+------------+
only showing top 20 rows


Then, we will remove any existing punctuation marks.


In [27]:

opinion_words_lower.select("opinion").filter(col("opinion").rlike(r"[.,!?;:]")).show()

+-------+
|opinion|
+-------+
+-------+



We don't actually see any matches of words with punctuation, that is because the data was generated using structured prompts that have resulted in already "clean" text, however its a good practice to account for this possibility.

In [29]:

opinion_words_cleaned = (opinion_words_lower
    .select(
        regexp_replace(col("opinion"), r"[,.!?:;]", "") # we use <r"[^a-zA-Z\s]"> to match any character that is not a letter (a-z or A-Z) or whitespace, and replace it with an empty string.
    .alias("clean_opinion_words"))
    .filter(col("clean_opinion_words") != ""))

opinion_words_cleaned.show()


+-------------------+
|clean_opinion_words|
+-------------------+
|             online|
|           learning|
|                 is|
|           flexible|
|                and|
|         convenient|
|             online|
|           learning|
|                 is|
|          effective|
|                for|
|       theory-based|
|           subjects|
|             online|
|           learning|
|                 is|
|          effective|
|                for|
|       theory-based|
|           subjects|
+-------------------+
only showing top 20 rows


In [32]:
final_word_count = opinion_words_cleaned.groupBy("clean_opinion_words").count().orderBy("count", ascending=False)
final_word_count.show()

+-------------------+-----+
|clean_opinion_words|count|
+-------------------+-----+
|             online|10000|
|           learning| 7909|
|                 is| 5951|
|          effective| 3975|
|             access| 2091|
|            diverse| 2091|
|            provide| 2091|
|          resources| 2091|
|          platforms| 2091|
|                 to| 2091|
|                for| 2016|
|       theory-based| 2016|
|           subjects| 2016|
|           flexible| 1976|
|                and| 1976|
|         convenient| 1976|
|            blended| 1959|
|               more| 1959|
|               than| 1959|
|              fully| 1959|
+-------------------+-----+
only showing top 20 rows


this includes a lot of Stop words so we will remove them using the nltk library.

In [35]:

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

opinion_words_cleaned_noStop = opinion_words_cleaned.filter(~col("clean_opinion_words").isin(stop_words))
opinion_words_cleaned_noStop.show()

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/martin/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


+-------------------+
|clean_opinion_words|
+-------------------+
|             online|
|           learning|
|           flexible|
|         convenient|
|             online|
|           learning|
|          effective|
|       theory-based|
|           subjects|
|             online|
|           learning|
|          effective|
|       theory-based|
|           subjects|
|             online|
|          platforms|
|            provide|
|             access|
|            diverse|
|          resources|
+-------------------+
only showing top 20 rows



Now we can gorup once again to get a better idea of how many distinct relevant words we have

In [36]:
final_word_count_noStop = opinion_words_cleaned_noStop.groupBy("clean_opinion_words").count().orderBy("count", ascending=False)
final_word_count_noStop.show()

+-------------------+-----+
|clean_opinion_words|count|
+-------------------+-----+
|             online|10000|
|           learning| 7909|
|          effective| 3975|
|             access| 2091|
|            diverse| 2091|
|            provide| 2091|
|          resources| 2091|
|          platforms| 2091|
|       theory-based| 2016|
|           subjects| 2016|
|           flexible| 1976|
|         convenient| 1976|
|            blended| 1959|
|              fully| 1959|
|    self-discipline| 1958|
|             strong| 1958|
|           requires| 1958|
+-------------------+-----+



Here we stablish a connection to the MongoDB server and initialize a new Database, with two collections, where we will inser first the raw data, translating every row of the original file into a JSON Document. The other one will be a collection where we isolate the opinion field, while mantaining the id field to connect this with the rest of the fields later on.

In [7]:
from pymongo import MongoClient

Mongo_URI = "mongodb://127.0.0.1:27017/"
client = MongoClient(Mongo_URI)

db = client["online_learning"]
raw_data = db["raw_records"]
opinion_col = db["opinions"]


In [40]:
raw_data.delete_many({})

for row in df.collect():
    record = row.asDict()
    record['_id'] = record.pop('respondent_id') # here we replace the automatically generated MongoDB index with the original index from the dataset
    raw_data.insert_one(record)

raw_data.count_documents({})

10000

In [41]:
raw_data.find_one()

{'_id': 1,
 'education_level': 'Undergraduate',
 'study_hours_per_day': 6,
 'preferred_learning_method': 'Practice exercises',
 'main_learning_challenge': 'Time management difficulty',
 'motivation_level': 'Low',
 'online_learning_opinion': 'Online learning is flexible and convenient',
 'device_used_for_study': 'Tablet'}

In [42]:
opinion_col.find_one()

{'_id': 1, 'opinion': 'Online learning is flexible and convenient'}